In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
import numpy as np
import cv2
import os
from tqdm import tqdm
import math
from skimage.metrics import peak_signal_noise_ratio as psnr
import matplotlib.pyplot as plt

# ===================================================================
# SECTION A: GREY WOLF OPTIMIZER FOR IMAGE ENHANCEMENT
# ===================================================================

def initialize_wolves(search_space, num_wolves):
    """Initializes the wolf population randomly within the search space."""
    dimensions = len(search_space)
    wolves = np.zeros((num_wolves, dimensions))
    for i in range(num_wolves):
        for j in range(dimensions):
            wolves[i, j] = np.random.uniform(search_space[j][0], search_space[j][1])
    return wolves

def gwo_algorithm(fitness_function, search_space, num_wolves, max_iterations, img_original):
    """Main GWO loop to find the best solution for a single image."""
    wolves = initialize_wolves(search_space, num_wolves)
    alpha_wolf, beta_wolf, delta_wolf = np.zeros(len(search_space)), np.zeros(len(search_space)), np.zeros(len(search_space))
    alpha_fitness, beta_fitness, delta_fitness = float('-inf'), float('-inf'), float('-inf')

    for i in range(num_wolves):
        fitness = fitness_function(wolves[i], img_original)
        if fitness > alpha_fitness:
            alpha_fitness, alpha_wolf = fitness, wolves[i].copy()
        elif fitness > beta_fitness:
            beta_fitness, beta_wolf = fitness, wolves[i].copy()
        elif fitness > delta_fitness:
            delta_fitness, delta_wolf = fitness, wolves[i].copy()

    for t in range(max_iterations):
        a = 2 - t * (2 / max_iterations)
        for i in range(num_wolves):
            for j in range(len(search_space)):
                r1, r2 = np.random.rand(2)
                A1, C1 = 2 * a * r1 - a, 2 * r2
                D_alpha = abs(C1 * alpha_wolf[j] - wolves[i, j])
                X1 = alpha_wolf[j] - A1 * D_alpha

                r1, r2 = np.random.rand(2)
                A2, C2 = 2 * a * r1 - a, 2 * r2
                D_beta = abs(C2 * beta_wolf[j] - wolves[i, j])
                X2 = beta_wolf[j] - A2 * D_beta

                r1, r2 = np.random.rand(2)
                A3, C3 = 2 * a * r1 - a, 2 * r2
                D_delta = abs(C3 * delta_wolf[j] - wolves[i, j])
                X3 = delta_wolf[j] - A3 * D_delta

                wolves[i, j] = (X1 + X2 + X3) / 3
                wolves[i, j] = np.clip(wolves[i, j], search_space[j][0], search_space[j][1])

            fitness = fitness_function(wolves[i], img_original)
            if fitness > alpha_fitness:
                alpha_fitness, alpha_wolf = fitness, wolves[i].copy()
            elif fitness > beta_fitness:
                beta_fitness, beta_wolf = fitness, wolves[i].copy()
            elif fitness > delta_fitness:
                delta_fitness, delta_wolf = fitness, wolves[i].copy()

    return alpha_wolf, alpha_fitness

def fitness_function(params, original_image_channel):
    """Calculates fitness (quality) of an enhanced image channel."""
    clip_limit = params[0]
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(8, 8))
    enhanced_image = clahe.apply(original_image_channel)

    hist = cv2.calcHist([enhanced_image], [0], None, [256], [0, 256])
    hist_norm = hist / (enhanced_image.size)
    image_entropy = -np.sum(hist_norm * np.log2(hist_norm + 1e-10))

    fitness = 0.6 * image_entropy + 0.4 * psnr(original_image_channel, enhanced_image)
    return fitness

def enhance_image_with_gwo(color_img):
    """
    Enhances a single color image using GWO to optimize CLAHE on the Lightness channel.
    """
    # GWO parameters (kept low for speed in this integrated notebook)
    search_space = np.array([[1.0, 10.0]]) # clipLimit range
    num_wolves = 10
    max_iterations = 20

    # Convert RGB to LAB color space to separate lightness from color
    lab_img = cv2.cvtColor(color_img, cv2.COLOR_RGB2LAB)
    l_channel, a_channel, b_channel = cv2.split(lab_img)

    # Run GWO to find the best clipLimit for the L-channel
    best_params, _ = gwo_algorithm(fitness_function, search_space, num_wolves, max_iterations, l_channel)
    best_clip_limit = best_params[0]

    # Apply CLAHE with the optimal parameter
    clahe = cv2.createCLAHE(clipLimit=best_clip_limit, tileGridSize=(8, 8))
    enhanced_l_channel = clahe.apply(l_channel)

    # Merge the enhanced L-channel with the original A and B channels
    merged_lab_img = cv2.merge([enhanced_l_channel, a_channel, b_channel])

    # Convert back to RGB color space
    enhanced_rgb_img = cv2.cvtColor(merged_lab_img, cv2.COLOR_LAB2RGB)

    return enhanced_rgb_img


# ===================================================================
# SECTION B: DEEP LEARNING MODEL TRAINING
# ===================================================================

# 1. Load and Pre-process Dataset
# =================================
DATA_DIR = "/content/drive/MyDrive/pepper"  # <-- change this to your dataset folder
IMG_SIZE = 224

def load_images(data_dir):
    images, labels = [], []
    classes = os.listdir(data_dir)
    class_to_idx = {cls: idx for idx, cls in enumerate(classes)}

    for cls in classes:
        cls_path = os.path.join(data_dir, cls)
        for img_file in tqdm(os.listdir(cls_path), desc=f"Loading & Enhancing {cls}"):
            img_path = os.path.join(cls_path, img_file)
            img = cv2.imread(img_path)
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                # =================================================
                img = enhance_image_with_gwo(img)
                # =================================================
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                images.append(img)
                labels.append(class_to_idx[cls])

    return np.array(images), np.array(labels), classes

X, y, classes = load_images(DATA_DIR)
X = X / 255.0  # normalize

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2. CBAM Attention Module
# ========================
class CBAM(layers.Layer):
    def __init__(self, ratio=8):
        super(CBAM, self).__init__()
        self.ratio = ratio

    def build(self, input_shape):
        self.channel = input_shape[-1]
        self.shared_dense_one = layers.Dense(self.channel // self.ratio, activation='relu')
        self.shared_dense_two = layers.Dense(self.channel)
        self.conv_spatial = layers.Conv2D(1, 7, padding='same', activation='sigmoid')

    def call(self, inputs):
        # Channel Attention
        avg_pool = tf.reduce_mean(inputs, axis=[1, 2], keepdims=True)
        max_pool = tf.reduce_max(inputs, axis=[1, 2], keepdims=True)
        avg_out = self.shared_dense_two(self.shared_dense_one(avg_pool))
        max_out = self.shared_dense_two(self.shared_dense_one(max_pool))
        scale = tf.nn.sigmoid(avg_out + max_out)
        channel_refined = inputs * scale

        # Spatial Attention
        avg_pool = tf.reduce_mean(channel_refined, axis=-1, keepdims=True)
        max_pool = tf.reduce_max(channel_refined, axis=-1, keepdims=True)
        concat = tf.concat([avg_pool, max_pool], axis=-1)
        spatial_out = self.conv_spatial(concat)
        refined_feature = channel_refined * spatial_out

        return refined_feature

# 3. Build Model
# ================
base_model = ResNet50(weights="imagenet", include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))

# Fine-tuning: freeze first layers, train last few blocks
for layer in base_model.layers[:140]:
    layer.trainable = False
for layer in base_model.layers[140:]:
    layer.trainable = True

inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = CBAM(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(len(classes), activation="softmax")(x)

model = models.Model(inputs, outputs)

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# 4. Training
# ===============================
history = model.fit(
    X_train, y_train,
    batch_size=32,
    validation_data=(X_val, y_val),
    epochs=80
)

# 5. Training Graphs
# =====================
# Accuracy graph
plt.figure(figsize=(8, 6))
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title("Model Accuracy", fontsize=14)
plt.xlabel("Epochs", fontsize=12)
plt.ylabel("Accuracy", fontsize=12)
plt.legend()
plt.grid(True)
plt.show()

# Loss graph
plt.figure(figsize=(8, 6))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title("Model Loss", fontsize=14)
plt.xlabel("Epochs", fontsize=12)
plt.ylabel("Loss", fontsize=12)
plt.legend()
plt.grid(True)
plt.show()

Loading & Enhancing Pepper__bell___healthy: 100%|██████████| 1478/1478 [06:23<00:00,  3.85it/s]
Loading & Enhancing Pepper__bell___Bacterial_spot: 100%|██████████| 997/997 [04:22<00:00,  3.79it/s]


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


ValueError: Only input tensors may be passed as positional arguments. The following argument value should be passed as a keyword argument: <CBAM name=cbam, built=False> (of type <class '__main__.CBAM'>)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns

# ===========================
# 1. Predictions
# ===========================
y_pred_probs = model.predict(X_val)
y_pred = np.argmax(y_pred_probs, axis=1)

# ===========================
# 2. Confusion Matrix
# ===========================
cm = confusion_matrix(y_val, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=classes, yticklabels=classes)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

# ===========================
# 3. Classification Report
# ===========================
print("Classification Report:\n")
print(classification_report(y_val, y_pred, target_names=classes))

# ===========================
# 4. Metrics
# ===========================
acc = accuracy_score(y_val, y_pred)
precision = precision_score(y_val, y_pred, average="weighted")
recall = recall_score(y_val, y_pred, average="weighted")
f1 = f1_score(y_val, y_pred, average="weighted")

print(f"✅ Accuracy: {acc:.4f}")
print(f"✅ Precision: {precision:.4f}")
print(f"✅ Recall: {recall:.4f}")
print(f"✅ F1-score: {f1:.4f}")
